## Deliverable 2. Create a Customer Travel Destinations Map.

In [1]:
# Dependencies and Setup
import pandas as pd
import requests
import gmaps

In [2]:
# Import API key
from config import g_key

# Configure gmaps API key
gmaps.configure(api_key=g_key)

In [3]:
# 1. Import the WeatherPy_database.csv file. 
city_data_df = pd.read_csv("../Weather_Database/WeatherPy_database.csv")
city_data_df.head()

,City_ID,City,Country,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Current Description
0,0,East London,ZA,-33.0153,27.9116,58.03,50,0,7.45,clear sky
1,1,Ponta Do Sol,PT,32.6667,-17.1000,67.01,66,34,7.63,scattered clouds
2,2,Yar-Sale,RU,66.8333,70.8333,53.51,48,100,27.40,overcast clouds
3,3,Garowe,SO,8.4054,48.4845,76.91,57,41,31.88,scattered clouds
4,4,Roccastrada,IT,43.0099,11.1665,74.01,90,16,1.57,few clouds


In [4]:
# 2. Prompt the user to enter minimum and maximum temperature criteria 

min_temp = float(input("What is the minimum temperature you would like for your trip? "))

max_temp = float(input("What is the maximum temperature you would like for your trip? "))


What is the minimum temperature you would like for your trip? 72
What is the maximum temperature you would like for your trip? 82


In [5]:
# 3. Filter the city_data_df DataFrame using the input statements to create a new DataFrame using the loc method.

preferred_cities_df = city_data_df.loc[(city_data_df["Max Temp"] <= max_temp) & \
                                       (city_data_df["Max Temp"] >= min_temp)]

preferred_cities_df.head(10)


,City_ID,City,Country,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Current Description
3,3,Garowe,SO,8.4054,48.4845,76.91,57,41,31.88,scattered clouds
4,4,Roccastrada,IT,43.0099,11.1665,74.01,90,16,1.57,few clouds
10,10,Acarau,BR,-2.8856,-40.1200,78.28,85,60,11.12,broken clouds
16,16,Piney Green,US,34.7160,-77.3202,80.55,96,100,3.44,overcast clouds
27,27,Kutum,SD,14.2000,24.6667,81.43,38,81,9.35,broken clouds
29,29,Barra,BR,-11.0894,-43.1417,76.33,54,0,17.67,clear sky
33,33,Hambantota,LK,6.1241,81.1185,78.98,80,91,18.45,overcast clouds
36,36,Georgetown,MY,5.4112,100.3354,80.53,86,20,7.00,thunderstorm
40,40,Maceio,BR,-9.6658,-35.7353,74.64,78,20,4.61,few clouds
45,45,Dzaoudzi,YT,-12.7887,45.2699,78.69,61,0,20.71,clear sky


In [6]:
# 4a. Determine if there are any empty rows.
preferred_cities_df.count()

City_ID                193
City                   193
Country                190
Lat                    193
Lng                    193
Max Temp               193
Humidity               193
Cloudiness             193
Wind Speed             193
Current Description    193
dtype: int64

In [7]:
# 4b. Drop any empty rows and create a new DataFrame that doesn’t have empty rows.
preferred_cities_clean_df = preferred_cities_df.dropna()
preferred_cities_clean_df.count()

City_ID                190
City                   190
Country                190
Lat                    190
Lng                    190
Max Temp               190
Humidity               190
Cloudiness             190
Wind Speed             190
Current Description    190
dtype: int64

In [8]:
# 5a. Create DataFrame called hotel_df to store hotel names along with city, country, max temp, and coordinates.
hotel_df = preferred_cities_clean_df[["City", "Country", "Max Temp", "Current Description", "Lat", "Lng"]].copy()

# 5b. Create a new column "Hotel Name"
hotel_df["Hotel Name"] = ""
hotel_df.tail(10)

,City,Country,Max Temp,Current Description,Lat,Lng,Hotel Name
684,Piranhas,BR,73.08,clear sky,-16.4269,-51.8222,
687,Ibb,YE,72.21,overcast clouds,13.9667,44.1833,
688,Domoni,KM,74.86,few clouds,-12.2569,44.5319,
691,Makurdi,NG,73.65,overcast clouds,7.7411,8.5121,
693,Khilchipur,IN,81.75,overcast clouds,24.0333,76.5667,
695,Juba,SS,72.54,overcast clouds,4.8517,31.5825,
696,Rock Sound,BS,80.28,overcast clouds,24.9000,-76.2000,
702,Nieuw Amsterdam,SR,80.76,scattered clouds,5.8833,-55.0833,
706,Moanda,GA,72.10,light rain,-1.5575,13.2178,
710,Pansemal,IN,78.49,overcast clouds,21.6500,74.7000,


In [9]:
# 6a. Set parameters to search for hotels with 5000 meters.
params = {
    "radius": 5000,
    "type": "lodging",
    "key": g_key
}

# 6b. Iterate through the hotel DataFrame.

for index, row in hotel_df.iterrows():
    
    # 6c. Get latitude and longitude from DataFrame

    lat = row["Lat"]
    lng = row["Lng"]
    
    params["location"] = f"{lat},{lng}"
    
    # 6d. Set up the base URL for the Google Directions API to get JSON data.
    
    base_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"

    # 6e. Make request and retrieve the JSON data from the search. 
   
    hotels = requests.get(base_url, params=params).json()
    
    # 6f. Get the first hotel from the results and store the name, if a hotel isn't found skip the city.
    
    try:
        hotel_df.loc[index, "Hotel Name"] = hotels["results"][0]["name"]
    except (IndexError):
        print("Hotel not found... skipping.")

Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.
Hotel not found... skipping.


In [10]:
# 7. Drop the rows where there is no Hotel Name.

hotel_df["Hotel Name"].astype(bool)

hotel_clean_df = hotel_df[hotel_df["Hotel Name"].astype(bool)]

hotel_clean_df.count()

City                   178
Country                178
Max Temp               178
Current Description    178
Lat                    178
Lng                    178
Hotel Name             178
dtype: int64

In [11]:
# 8a. Create the output File (CSV)

output_data_file = "../Vacation_Search/WeatherPy_vacation.csv"

# 8b. Export the City_Data into a csv
hotel_clean_df.to_csv(output_data_file, index_label="City_ID")

In [12]:
# 9. Using the template add city name, the country code, the weather description and maximum temperature for the city.
info_box_template = """

<dl>

<dt>Hotel Name</dt><dd>{Hotel Name}</dd>

<dt>City</dt><dd>{City}</dd>

<dt>Country</dt><dd>{Country}</dd>

<dt>Current Weather</dt><dd>{Current Description} and {Max Temp} °F</dd>


</dl>

"""

# 10a. Get the data from each row and add it to the formatting template and store the data in a list.
hotel_info = [info_box_template.format(**row) for index, row in hotel_clean_df.iterrows()]

# 10b. Get the latitude and longitude from each row and store in a new DataFrame.
locations = hotel_clean_df[["Lat", "Lng"]]

In [13]:
# 11a. Add a marker layer for each city to the map. 

# Add a marker layer map for the vacation spots and a pop-up marker for each city.

locations = hotel_clean_df[["Lat", "Lng"]]

max_temp = hotel_clean_df["Max Temp"]

fig = gmaps.figure(center=(30.0, 31.0), zoom_level=1.5)

marker_layer = gmaps.marker_layer(locations, info_box_content=hotel_info)

fig.add_layer(marker_layer)


# 11b. Display the figure

fig


Figure(layout=FigureLayout(height='420px'))